# 🎭 Wan2.2 Animate — Reemplazo de Personaje en Video

Transfiere los movimientos y gestos de un personaje en un **video existente** a tu **imagen de referencia**.  
Basado en [Wan2.2-Animate-14B](https://github.com/Wan-Video/Wan2.2) · Tencent / Wan-AI.

## Dos modos disponibles
| Modo | Descripción |
|------|-------------|
| **Reemplazo** (`replace`) | Tu imagen sustituye al personaje del video, adoptando sus movimientos exactos |
| **Animación** (`animate`) | Tu imagen se anima con los movimientos del video de referencia |

## Flujo de trabajo
1. Verificar GPU y fijar NumPy
2. Clonar repo e instalar dependencias
3. Descargar modelo Wan2.2-Animate-14B
4. Subir tu imagen y video de referencia
5. Preprocesar materiales
6. Generar el video

---
> ⚠️ **GPU requerida**: Este modelo necesita **A100 (40 GB)** para el modo `replace`.  
> En T4 (16 GB) puedes intentar el modo `animate` reduciendo la resolución a 832×480.

## Paso 0 — Verificar GPU y fijar NumPy

In [ ]:
import subprocess, sys

# Verificar GPU
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '❌ GPU no detectada.')

# Wan2.2 requiere numpy>=1.23.5,<2  y  torch>=2.4
np_ver = subprocess.run(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
    capture_output=True, text=True).stdout.strip()
print(f'NumPy instalado: {np_ver}')

if np_ver and int(np_ver.split('.')[0]) >= 2:
    print('⬇️  Fijando NumPy 1.26.4...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==1.26.4', '--force-reinstall'], check=True)
    print('✅ NumPy 1.26.4 instalado — reiniciando kernel...')
    import os; os.kill(os.getpid(), 9)   # reinicio limpio
else:
    print(f'✅ NumPy {np_ver} — OK')

## Paso 1 — Clonar repo e instalar dependencias

> ⏳ ~5 min la primera vez. `flash_attn` tarda más porque compila desde fuente.

In [ ]:
import os, subprocess, sys

WAN_DIR = '/content/Wan2.2'

if not os.path.exists(WAN_DIR):
    print('📦 Clonando Wan2.2...')
    subprocess.run(['git', 'clone', 'https://github.com/Wan-Video/Wan2.2.git', WAN_DIR], check=True)
else:
    print('✅ Wan2.2 ya clonado')

os.chdir(WAN_DIR)
print(f'📁 {os.getcwd()}')

In [ ]:
import subprocess, sys

# Wan2.2 requiere torch>=2.4. Colab trae 2.2 — actualizamos.
print('⬇️  Instalando torch 2.4+ con CUDA 12.1...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.4.0', 'torchvision>=0.19.0', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
], check=True)

print('⬇️  Instalando dependencias de Wan2.2...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.31.0',
    'transformers>=4.49.0,<=4.51.3',
    'tokenizers>=0.20.3',
    'accelerate>=1.1.1',
    'opencv-python-headless>=4.9.0.80',
    'imageio[ffmpeg]', 'imageio-ffmpeg',
    'easydict', 'ftfy', 'tqdm',
    'huggingface_hub',
    'numpy>=1.23.5,<2',    # fijar numpy<2 al final
], check=True)

# flash_attn: instalar wheel precompilado (mucho más rápido que compilar)
print('⬇️  Instalando flash_attn (wheel precompilado)...')
fa_url = (
    'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/'
    'flash_attn-2.7.4.post1+cu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl'
)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', fa_url],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  flash_attn wheel falló, instalando con pip (lento ~10 min)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flash_attn',
                    '--no-build-isolation'], check=True)
print('✅ flash_attn instalado')

# Verificar entorno
import torch, numpy as np
print(f'\n✅ torch {torch.__version__} | CUDA {torch.cuda.is_available()} | NumPy {np.__version__}')
assert np.__version__.startswith('1.'), f'❌ NumPy debe ser 1.x, tienes {np.__version__}'
assert torch.__version__ >= '2.4', f'❌ torch debe ser >=2.4, tienes {torch.__version__}'
print('✅ Entorno listo')

## Paso 2 — Autenticación en Hugging Face

El modelo Wan2.2-Animate-14B puede requerir aceptar términos.  
1. Crea un token en: https://huggingface.co/settings/tokens  
2. Pégalo abajo o guárdalo como secreto `HF_TOKEN` en Colab.

In [ ]:
import os
from huggingface_hub import login, whoami

HF_TOKEN = ""  # @param {type:"string"}

# Intentar desde secretos de Colab si no se pegó token
if not HF_TOKEN.strip():
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN.strip()
    print(f'✅ Autenticado como: {whoami()["name"]}')
else:
    print('⚠️  Sin token — el modelo podría no descargarse si es privado/gated')

## Paso 3 — Descargar modelo Wan2.2-Animate-14B

> ⏳ ~30–60 min la primera vez (~30 GB). Los modelos se cachean en `/content/models/`.

In [ ]:
import os, subprocess, sys

MODEL_DIR = '/content/models/Wan2.2-Animate-14B'
os.makedirs(MODEL_DIR, exist_ok=True)

# Verificar si ya está descargado (busca archivo clave)
already_downloaded = os.path.exists(os.path.join(MODEL_DIR, 'config.json'))

if not already_downloaded:
    print(f'⬇️  Descargando Wan2.2-Animate-14B a {MODEL_DIR} (~30 GB)...')
    subprocess.run([
        sys.executable, '-m', 'huggingface_hub', 'download',
        '--repo-id', 'Wan-AI/Wan2.2-Animate-14B',
        '--local-dir', MODEL_DIR,
        '--local-dir-use-symlinks', 'False',
    ], check=True, env={**os.environ})
    print('✅ Modelo descargado')
else:
    print(f'✅ Modelo ya en {MODEL_DIR}')

print('\n📦 Contenido del modelo:')
!ls -lh {MODEL_DIR}/ | head -20

## Paso 4 — Subir archivos de entrada

Necesitas:
- **Imagen del personaje** (`.jpg` / `.png`) — el personaje que reemplazará al del video
- **Video de referencia** (`.mp4`) — el video con el movimiento a transferir

> **Consejos para mejores resultados:**
> - Imagen: cuerpo completo, fondo liso, buena iluminación, persona de frente
> - Video: persona claramente visible, cuerpo completo en frame, iluminación uniforme

In [ ]:
from google.colab import files
import os, shutil
from PIL import Image
import IPython.display as display

os.makedirs('/content/inputs', exist_ok=True)

print('📸 Sube la IMAGEN del personaje (jpg/png):')
uploaded_img = files.upload()

ref_image_path = None
for fname, data in uploaded_img.items():
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f: f.write(data)
    ref_image_path = dest
    print(f'✅ Imagen guardada: {ref_image_path}')

if ref_image_path:
    img = Image.open(ref_image_path)
    print(f'   Tamaño: {img.size[0]}x{img.size[1]} px')
    display.display(img.resize((200, int(200 * img.size[1] / img.size[0]))))

In [ ]:
import cv2

print('🎥 Sube el VIDEO DE REFERENCIA (mp4):')
uploaded_vid = files.upload()

ref_video_path = None
for fname, data in uploaded_vid.items():
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f: f.write(data)
    ref_video_path = dest
    print(f'✅ Video guardado: {ref_video_path}')

if ref_video_path:
    cap = cv2.VideoCapture(ref_video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_v = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    dur = total / fps_v if fps_v > 0 else 0
    print(f'   {w}x{h} | {total} frames | {fps_v:.1f} fps | {dur:.1f}s')
    if dur > 15:
        print(f'⚠️  Video largo ({dur:.0f}s). Considera recortarlo a <10s para ahorrar VRAM.')

## Paso 5 — Configurar modo y parámetros

In [ ]:
# ============================================================
#  MODO DE OPERACIÓN
# ============================================================

# "replace" = tu imagen REEMPLAZA al personaje del video (mantiene fondo original)
# "animate" = tu imagen SE ANIMA con los movimientos del video (fondo neutro)
MODE = "replace"  # @param ["replace", "animate"]

# ============================================================
#  RESOLUCIÓN DE SALIDA
# ============================================================
# Recomendado para A100 (40GB): 1280x720
# Para T4 (16GB) modo animate: 832x480
OUTPUT_WIDTH  = 1280  # @param {type:"slider", min:480, max:1280, step:64}
OUTPUT_HEIGHT = 720   # @param {type:"slider", min:272, max:720, step:64}

# FPS de procesamiento del video (-1 = usar FPS original)
PROC_FPS = 30  # @param {type:"slider", min:-1, max:30, step:1}

# ============================================================
#  PARÁMETROS MODO REPLACE
# ============================================================
REPLACE_ITERATIONS = 3   # @param {type:"slider", min:1, max:5, step:1}
REPLACE_KERNEL_K   = 7   # @param {type:"slider", min:3, max:15, step:2}
REPLACE_W_LEN      = 1   # @param {type:"slider", min:1, max:4, step:1}
REPLACE_H_LEN      = 1   # @param {type:"slider", min:1, max:4, step:1}

# ============================================================
#  PARÁMETROS MODO ANIMATE
# ============================================================
USE_RETARGET = True   # @param {type:"boolean"} — reorienta la pose a la imagen
USE_FLUX     = False  # @param {type:"boolean"} — edición de imagen para poses no estándar

# ============================================================
#  INFERENCIA
# ============================================================
REFERT_NUM        = 1      # @param {type:"slider", min:1, max:4, step:1}
USE_RELIGHTING    = True   # @param {type:"boolean"} — mejora la iluminación (solo replace)
OFFLOAD_MODEL     = True   # @param {type:"boolean"} — libera capas a CPU (ahorra VRAM)

# Rutas
WAN_DIR    = '/content/Wan2.2'
MODEL_DIR  = '/content/models/Wan2.2-Animate-14B'
PROC_DIR   = '/content/process_results'
OUTPUT_DIR = '/content/outputs'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'⚙️  Modo        : {MODE.upper()}')
print(f'   Resolución  : {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}')
print(f'   FPS proceso : {PROC_FPS}')
print(f'   Offload     : {OFFLOAD_MODEL}')
if MODE == 'replace':
    print(f'   Iterations  : {REPLACE_ITERATIONS} | k={REPLACE_KERNEL_K} | w={REPLACE_W_LEN} | h={REPLACE_H_LEN}')
    print(f'   Relighting  : {USE_RELIGHTING}')
else:
    print(f'   Retarget    : {USE_RETARGET} | Flux: {USE_FLUX}')

## Paso 6 — Preprocesar materiales

> ⏳ Este paso extrae la pose del video y adapta tu imagen. Toma ~2–5 min.

In [ ]:
import os, subprocess, sys, shutil

assert ref_image_path and os.path.exists(ref_image_path), \
    '❌ Imagen no encontrada. Ejecuta el Paso 4.'
assert ref_video_path and os.path.exists(ref_video_path), \
    '❌ Video no encontrado. Ejecuta el Paso 4.'

# Limpiar resultados anteriores
if os.path.exists(PROC_DIR):
    shutil.rmtree(PROC_DIR)
os.makedirs(PROC_DIR)

ckpt_path = os.path.join(MODEL_DIR, 'process_checkpoint')
preprocess_script = os.path.join(
    WAN_DIR, 'wan', 'modules', 'animate', 'preprocess', 'preprocess_data.py'
)

cmd = [
    sys.executable, preprocess_script,
    '--ckpt_path',      ckpt_path,
    '--video_path',     ref_video_path,
    '--refer_path',     ref_image_path,
    '--save_path',      PROC_DIR,
    '--resolution_area', str(OUTPUT_WIDTH), str(OUTPUT_HEIGHT),
    '--fps',            str(PROC_FPS),
]

if MODE == 'replace':
    cmd += [
        '--replace_flag',
        '--iterations', str(REPLACE_ITERATIONS),
        '--k',          str(REPLACE_KERNEL_K),
        '--w_len',      str(REPLACE_W_LEN),
        '--h_len',      str(REPLACE_H_LEN),
    ]
else:  # animate
    if USE_RETARGET:
        cmd.append('--retarget_flag')
    if USE_FLUX:
        cmd.append('--use_flux')

print('🔧 Preprocesando...')
print('Comando:', ' '.join(cmd))
print('─' * 60)

env = {**os.environ, 'PYTHONPATH': WAN_DIR}
result = subprocess.run(cmd, cwd=WAN_DIR, env=env)

if result.returncode != 0:
    print('❌ Error en preprocesamiento.')
else:
    print('\n✅ Preprocesamiento completado')
    print('Archivos generados:')
    for root, dirs, files_list in os.walk(PROC_DIR):
        for f in files_list:
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1e6
            print(f'  {path} ({size:.1f} MB)')

## Paso 7 — Generar video

> ⏳ ~10–30 min dependiendo de la longitud del video y la GPU.  
> La primera vez descarga pesos adicionales del modelo.

In [ ]:
import os, subprocess, sys

generate_script = os.path.join(WAN_DIR, 'generate.py')

cmd = [
    sys.executable, generate_script,
    '--task',          'animate-14B',
    '--ckpt_dir',      MODEL_DIR,
    '--src_root_path', PROC_DIR,
    '--save_file',     os.path.join(OUTPUT_DIR, 'output.mp4'),
    '--refert_num',    str(REFERT_NUM),
]

if MODE == 'replace':
    cmd.append('--replace_flag')
    if USE_RELIGHTING:
        cmd.append('--use_relighting_lora')

if OFFLOAD_MODEL:
    cmd.append('--offload_model')
    cmd.append('True')

env = {
    **os.environ,
    'PYTHONPATH': WAN_DIR,
    'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:256',
}

print('🚀 Generando video...')
print('Comando:', ' '.join(cmd))
print('─' * 60)

result = subprocess.run(cmd, cwd=WAN_DIR, env=env)

if result.returncode != 0:
    print('❌ Error en la generación. Revisa los logs arriba.')
else:
    print('\n✅ Video generado')
    !ls -lh {OUTPUT_DIR}/

## Paso 8 — Ver y descargar el resultado

In [ ]:
import glob, os
from IPython.display import HTML, display as ipy_display
from base64 import b64encode

videos = sorted(glob.glob(f'{OUTPUT_DIR}/**/*.mp4', recursive=True),
                key=os.path.getmtime, reverse=True)

if not videos:
    print('❌ No se encontró video de salida.')
else:
    latest = videos[0]
    size_mb = os.path.getsize(latest) / 1e6
    print(f'✅ Video: {latest} ({size_mb:.1f} MB)')

    with open(latest, 'rb') as f:
        b64 = b64encode(f.read()).decode()
    ipy_display(HTML(f'''
    <video controls width="640">
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))

In [ ]:
from google.colab import files
if videos:
    print(f'⬇️  Descargando {os.path.basename(latest)}...')
    files.download(latest)
else:
    print('❌ Sin video para descargar.')

---
## Referencia rápida

### Diferencias entre modos

| | **replace** | **animate** |
|---|---|---|
| Fondo | Conserva el fondo original del video | Fondo neutro/generado |
| Resultado | Reemplazo realista in-situ | Personaje animado |
| VRAM mín. | ~40 GB (A100) | ~16 GB (T4, res. baja) |
| Velocidad | Más lento | Más rápido |

### Parámetros modo `replace`
| Parámetro | Efecto |
|---|---|
| `iterations` | Mayor = bordes más limpios en la máscara |
| `k` (kernel) | Mayor = dilación de máscara más amplia |
| `w_len / h_len` | Subdivisiones del contorno para más detalle |
| `use_relighting_lora` | Ajusta iluminación del personaje al fondo |

### Solución de problemas
- **OOM en A100**: activa `offload_model=True` y baja la resolución a 1024×576
- **OOM en T4**: usa modo `animate` con resolución 832×480
- **Pose incorrecta**: activa `retarget_flag=True` (solo modo animate)
- **Video largo**: recorta a <10 segundos con `ffmpeg -ss 0 -t 10 -i input.mp4 out.mp4`